# Langkah 1 Anggota A — Unduh dan Audit Dataset RAVDESS

Notebook ini menjalankan langkah kerja 1 dari rencana kerja Proyek 07:

1. Menyiapkan folder proyek di Google Drive
2. Mengunduh bagian speech RAVDESS dari Zenodo (±208 MB)
3. Mengekstrak file zip
4. Membaca informasi dari nama setiap file WAV
5. Mengaudit data (jumlah file, kelas, aktor, gender, sample rate, durasi)
6. Menyimpan `metadata.csv` dan `audit_report.txt`

**Cara pakai:** jalankan sel satu per satu dari atas ke bawah (klik sel, tekan `Shift + Enter`). Baca penjelasan di atas setiap sel sebelum menjalankannya.

## Sel 1 — Siapkan folder proyek

Sel ini menghubungkan Colab ke Google Drive Anda, lalu membuat folder `MyDrive/project07/data/raw`.
Data disimpan di Drive supaya **tidak hilang** saat sesi Colab terputus, dan bisa dibagikan ke anggota B dan C.

Saat dijalankan, Colab akan meminta izin akses Google Drive: pilih akun Anda lalu klik **Izinkan**.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/project07'
except ImportError:
    # Jika dijalankan di komputer sendiri (bukan Colab), folder dibuat di direktori saat ini
    BASE_DIR = os.path.abspath('project07')

DATA_DIR = os.path.join(BASE_DIR, 'data')
RAW_DIR = os.path.join(DATA_DIR, 'raw')
AUDIO_DIR = os.path.join(RAW_DIR, 'ravdess_speech')
os.makedirs(AUDIO_DIR, exist_ok=True)

print('Folder proyek :', BASE_DIR)
print('Folder audio  :', AUDIO_DIR)

Mounted at /content/drive
Folder proyek : /content/drive/MyDrive/project07
Folder audio  : /content/drive/MyDrive/project07/data/raw/ravdess_speech


## Sel 2 — Unduh dataset dari Zenodo

Yang diunduh hanya **Audio_Speech_Actors_01-24.zip**: rekaman suara bicara saja, tanpa video dan lagu.
Jika file sudah pernah diunduh, sel ini akan melewatinya.

Jika unduhan gagal, unduh manual dari https://zenodo.org/records/1188976 (cari file `Audio_Speech_Actors_01-24.zip`),
lalu unggah ke folder `MyDrive/project07/data/raw/` di Google Drive.

In [ ]:
import urllib.request
import zipfile

ZIP_URL = 'https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1'
ZIP_PATH = os.path.join(RAW_DIR, 'Audio_Speech_Actors_01-24.zip')

def tampilkan_progres(blok, ukuran_blok, total):
    selesai = blok * ukuran_blok
    if total > 0 and blok % 200 == 0:
        print(f'  {selesai / 1e6:6.1f} / {total / 1e6:.1f} MB', end='\r')

if os.path.exists(ZIP_PATH):
    print('File zip sudah ada, unduhan dilewati.')
else:
    print('Mengunduh... (biasanya 1–5 menit)')
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH, tampilkan_progres)
    print()

ukuran_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f'Ukuran file: {ukuran_mb:.1f} MB')

# Cek bahwa file zip tidak rusak
with zipfile.ZipFile(ZIP_PATH) as z:
    rusak = z.testzip()
print('Zip utuh.' if rusak is None else f'Zip RUSAK pada file: {rusak} — hapus file zip dan unduh ulang.')

Mengunduh... (biasanya 1–5 menit)
   208.1 / 208.5 MB
Ukuran file: 208.5 MB
Zip utuh.


## Sel 3 — Ekstrak file zip

Isi zip berupa 24 folder (`Actor_01` sampai `Actor_24`), masing-masing berisi 60 file WAV.

In [ ]:
import glob

with zipfile.ZipFile(ZIP_PATH) as z:
    isi_zip = [n for n in z.namelist() if n.lower().endswith('.wav')]
    sudah_ada = glob.glob(os.path.join(AUDIO_DIR, '**', '*.wav'), recursive=True)
    if len(sudah_ada) >= len(isi_zip):
        print('File sudah diekstrak sebelumnya, dilewati.')
    else:
        print('Mengekstrak...')
        z.extractall(AUDIO_DIR)

wav_files = sorted(glob.glob(os.path.join(AUDIO_DIR, '**', '*.wav'), recursive=True))
print('Jumlah file WAV ditemukan:', len(wav_files))
print('Contoh:', os.path.relpath(wav_files[0], AUDIO_DIR))

Mengekstrak...
Jumlah file WAV ditemukan: 1440
Contoh: Actor_01/03-01-01-01-01-01-01.wav


## Sel 4 — Baca informasi dari nama file

Setiap nama file RAVDESS terdiri dari 7 angka, contoh `03-01-06-01-02-01-12.wav`:

| Posisi | Arti | Nilai |
| --- | --- | --- |
| 1 | Modality | 03 = audio saja |
| 2 | Vocal channel | 01 = speech (bicara) |
| 3 | Emotion | 01 neutral, 02 calm, 03 happy, 04 sad, 05 angry, 06 fearful, 07 disgust, 08 surprised |
| 4 | Intensity | 01 normal, 02 strong (neutral tidak punya strong) |
| 5 | Statement | 01 "Kids are talking by the door", 02 "Dogs are sitting by the door" |
| 6 | Repetition | 01 atau 02 |
| 7 | Actor | 01–24; ganjil = laki-laki, genap = perempuan |

Sel ini mengubah setiap nama file menjadi satu baris tabel, ditambah durasi, sample rate, dan jumlah channel yang dibaca dari file audionya.

In [ ]:
import pandas as pd
import soundfile as sf

EMOTION = {'01': 'neutral', '02': 'calm', '03': 'happy', '04': 'sad',
           '05': 'angry', '06': 'fearful', '07': 'disgust', '08': 'surprised'}
INTENSITY = {'01': 'normal', '02': 'strong'}

def baca_metadata(path):
    nama = os.path.basename(path)
    bagian = nama[:-4].split('-')
    if len(bagian) != 7:
        raise ValueError(f'Nama file tidak sesuai format: {nama}')
    modality, channel, emo, inten, statement, rep, actor = bagian
    info = sf.info(path)
    folder = os.path.basename(os.path.dirname(path))
    return {
        'file': os.path.relpath(path, AUDIO_DIR).replace(os.sep, '/'),
        'folder_actor': int(folder.split('_')[-1]) if folder.startswith('Actor_') else int(actor),
        'modality': modality,
        'channel': channel,
        'emotion': EMOTION[emo],
        'emotion_code': int(emo),
        'intensity': INTENSITY[inten],
        'statement': int(statement),
        'repetition': int(rep),
        'actor': int(actor),
        'gender': 'male' if int(actor) % 2 == 1 else 'female',
        'duration_s': round(info.duration, 3),
        'sample_rate': info.samplerate,
        'channels': info.channels,
    }

baris, gagal = [], []
for p in wav_files:
    try:
        baris.append(baca_metadata(p))
    except Exception as e:
        gagal.append((p, str(e)))

meta = pd.DataFrame(baris).sort_values(['actor', 'emotion_code', 'intensity', 'statement', 'repetition']).reset_index(drop=True)
print('Berhasil dibaca:', len(meta), '| Gagal:', len(gagal))
for p, e in gagal[:10]:
    print('  GAGAL', p, '→', e)
meta.head(10)

Berhasil dibaca: 1440 | Gagal: 0


,file,folder_actor,modality,channel,emotion,emotion_code,intensity,statement,repetition,actor,gender,duration_s,sample_rate,channels
0,Actor_01/03-01-01-01-01-01-01.wav,1,03,01,neutral,1,normal,1,1,1,male,3.303,48000,1
1,Actor_01/03-01-01-01-01-02-01.wav,1,03,01,neutral,1,normal,1,2,1,male,3.337,48000,1
2,Actor_01/03-01-01-01-02-01-01.wav,1,03,01,neutral,1,normal,2,1,1,male,3.270,48000,1
3,Actor_01/03-01-01-01-02-02-01.wav,1,03,01,neutral,1,normal,2,2,1,male,3.170,48000,1
4,Actor_01/03-01-02-01-01-01-01.wav,1,03,01,calm,2,normal,1,1,1,male,3.537,48000,1
5,Actor_01/03-01-02-01-01-02-01.wav,1,03,01,calm,2,normal,1,2,1,male,3.604,48000,2
6,Actor_01/03-01-02-01-02-01-01.wav,1,03,01,calm,2,normal,2,1,1,male,3.503,48000,1
7,Actor_01/03-01-02-01-02-02-01.wav,1,03,01,calm,2,normal,2,2,1,male,3.470,48000,1
8,Actor_01/03-01-02-02-01-01-01.wav,1,03,01,calm,2,strong,1,1,1,male,3.704,48000,1
9,Actor_01/03-01-02-02-01-02-01.wav,1,03,01,calm,2,strong,1,2,1,male,4.004,48000,1


## Sel 5 — Audit data

Sel ini memeriksa apakah data sesuai dengan deskripsi resmi RAVDESS. Setiap baris hasil diberi tanda **OK** atau **CEK**.
Jika ada **CEK**, catat dan kabari kelompok sebelum lanjut ke langkah berikutnya.

In [ ]:
laporan = []

def cek(judul, kondisi, detail=''):
    tanda = 'OK ' if kondisi else 'CEK'
    teks = f'[{tanda}] {judul}' + (f' — {detail}' if detail else '')
    laporan.append(teks)
    print(teks)

per_emosi = meta['emotion'].value_counts()
per_aktor = meta['actor'].value_counts()
aktor_gender = meta.groupby('gender')['actor'].nunique()

cek('Total file = 1.440', len(meta) == 1440, f'{len(meta)} file')
cek('Tidak ada file gagal dibaca', len(gagal) == 0, f'{len(gagal)} gagal')
cek('Ada 24 aktor', meta['actor'].nunique() == 24, f"{meta['actor'].nunique()} aktor")
cek('Setiap aktor punya 60 file', (per_aktor == 60).all(), f'min {per_aktor.min()}, maks {per_aktor.max()}')
cek('Gender seimbang 12 L / 12 P', aktor_gender.get('male', 0) == 12 and aktor_gender.get('female', 0) == 12, aktor_gender.to_dict())
cek('Semua file audio-only speech (03-01)', ((meta['modality'] == '03') & (meta['channel'] == '01')).all())
cek('Nomor aktor di nama file = nomor folder', (meta['actor'] == meta['folder_actor']).all())
cek('Neutral = 96 file', per_emosi.get('neutral', 0) == 96, f"{per_emosi.get('neutral', 0)} file")
lain = per_emosi.drop('neutral', errors='ignore')
cek('7 emosi lain masing-masing 192 file', (lain == 192).all() and len(lain) == 7, lain.to_dict())
cek('Neutral hanya intensitas normal', (meta.loc[meta['emotion'] == 'neutral', 'intensity'] == 'normal').all())
cek('Tidak ada nama file ganda', not meta['file'].str.split('/').str[-1].duplicated().any())
cek('Sample rate seragam', meta['sample_rate'].nunique() == 1, f"{sorted(int(x) for x in meta['sample_rate'].unique())} Hz")
cek('Jumlah channel seragam', meta['channels'].nunique() == 1, f"{sorted(int(x) for x in meta['channels'].unique())} channel")

d = meta['duration_s']
ringkas_durasi = f'min {d.min():.2f} s, rata-rata {d.mean():.2f} s, maks {d.max():.2f} s, total {d.sum() / 60:.1f} menit'
laporan.append('Durasi: ' + ringkas_durasi)
print('Durasi:', ringkas_durasi)

[OK ] Total file = 1.440 — 1440 file
[OK ] Tidak ada file gagal dibaca — 0 gagal
[OK ] Ada 24 aktor — 24 aktor
[OK ] Setiap aktor punya 60 file — min 60, maks 60
[OK ] Gender seimbang 12 L / 12 P — {'female': 12, 'male': 12}
[OK ] Semua file audio-only speech (03-01)
[OK ] Nomor aktor di nama file = nomor folder
[OK ] Neutral = 96 file — 96 file
[OK ] 7 emosi lain masing-masing 192 file — {'calm': 192, 'happy': 192, 'sad': 192, 'angry': 192, 'disgust': 192, 'fearful': 192, 'surprised': 192}
[OK ] Neutral hanya intensitas normal
[OK ] Tidak ada nama file ganda
[OK ] Sample rate seragam — [48000] Hz
[CEK] Jumlah channel seragam — [1, 2] channel
Durasi: min 2.94 s, rata-rata 3.70 s, maks 5.27 s, total 88.8 menit


## Sel 6 — Tabel ringkasan untuk dataset card

Tabel-tabel ini akan Anda pakai di langkah 2 (menulis dataset card) dan di bagian Method mini-paper.

In [ ]:
print('Jumlah file per emosi dan intensitas:')
display(pd.crosstab(meta['emotion'], meta['intensity'], margins=True, margins_name='total'))

print('Jumlah file per emosi dan gender:')
display(pd.crosstab(meta['emotion'], meta['gender'], margins=True, margins_name='total'))

print('Durasi (detik) per emosi:')
display(meta.groupby('emotion')['duration_s'].agg(['count', 'mean', 'min', 'max']).round(2))

Jumlah file per emosi dan intensitas:


intensity,normal,strong,total
emotion,,,
angry,96,96,192
calm,96,96,192
disgust,96,96,192
fearful,96,96,192
happy,96,96,192
neutral,96,0,96
sad,96,96,192
surprised,96,96,192
total,768,672,1440


Jumlah file per emosi dan gender:


gender,female,male,total
emotion,,,
angry,96,96,192
calm,96,96,192
disgust,96,96,192
fearful,96,96,192
happy,96,96,192
neutral,48,48,96
sad,96,96,192
surprised,96,96,192
total,720,720,1440


Durasi (detik) per emosi:


,count,mean,min,max
emotion,,,,
angry,192,3.87,3.24,5.11
calm,192,3.80,2.94,4.77
disgust,192,3.94,3.14,5.27
fearful,192,3.57,3.07,5.00
happy,192,3.64,3.10,4.40
neutral,96,3.50,3.07,4.14
sad,192,3.69,3.10,4.74
surprised,192,3.49,2.97,4.64


## Sel 7 — Simpan hasil

Dua file disimpan ke `MyDrive/project07/data/`:

- `metadata.csv` — satu baris per file audio. File ini dipakai di langkah ekstraksi fitur dan pembuatan split.
- `audit_report.txt` — hasil audit untuk dilampirkan di dataset card.

In [ ]:
META_PATH = os.path.join(DATA_DIR, 'metadata.csv')
REPORT_PATH = os.path.join(DATA_DIR, 'audit_report.txt')

kolom = ['file', 'actor', 'gender', 'emotion', 'emotion_code', 'intensity',
         'statement', 'repetition', 'duration_s', 'sample_rate', 'channels']
meta[kolom].to_csv(META_PATH, index=False)

with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write('Audit RAVDESS speech (Audio_Speech_Actors_01-24)\n')
    f.write('Sumber: https://zenodo.org/records/1188976 | Lisensi: CC BY-NC-SA 4.0\n\n')
    f.write('\n'.join(laporan) + '\n')

print('Tersimpan:', META_PATH)
print('Tersimpan:', REPORT_PATH)

Tersimpan: /content/drive/MyDrive/project07/data/metadata.csv
Tersimpan: /content/drive/MyDrive/project07/data/audit_report.txt


## Sel 8 — Dengarkan beberapa contoh

Dengarkan satu contoh tiap emosi dari aktor yang sama. Ini membantu Anda memahami perbedaan yang nanti harus ditangkap oleh fitur akustik.

In [ ]:
from IPython.display import Audio, display

contoh = meta[(meta['actor'] == 1) & (meta['statement'] == 1) & (meta['repetition'] == 1)
              & (meta['intensity'].isin(['normal']))]
for _, r in contoh.iterrows():
    print(r['emotion'], '—', r['file'])
    display(Audio(os.path.join(AUDIO_DIR, r['file'])))

Output hidden; open in https://colab.research.google.com to view.

## Selesai — langkah berikutnya

1. Pastikan semua baris audit bertanda **OK**. Jika ada **CEK**, kirim isi `audit_report.txt` ke grup.
2. Umumkan ke grup bahwa `metadata.csv` sudah ada di Drive bersama.
3. Lanjut ke **langkah 2: dataset card**, memakai tabel di Sel 6 dan isi `audit_report.txt`.

CEK: 5 file stereo

Hampir semua rekaman bersifat mono (satu channel), tapi ada 5 file yang tersimpan sebagai stereo (dua channel):

File	Aktor	Emosi
Actor_01/03-01-02-01-01-02-01.wav	1 (L)	calm, normal
Actor_01/03-01-08-01-02-02-01.wav	1 (L)	surprised, normal
Actor_05/03-01-02-01-02-02-05.wav	5 (L)	calm, normal
Actor_20/03-01-03-01-02-01-20.wav	20 (P)	happy, normal
Actor_20/03-01-06-01-01-02-20.wav	20 (P)	fearful, normal

In [2]:
import os
import numpy as np
import pandas as pd
import soundfile as sf

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/project07'
except ImportError:
    BASE_DIR = os.path.abspath('project07')

AUDIO_DIR = os.path.join(BASE_DIR, 'data', 'raw', 'ravdess_speech')
meta = pd.read_csv(os.path.join(BASE_DIR, 'data', 'metadata.csv'))

stereo = meta.loc[meta['channels'] == 2, 'file']
print('Jumlah file stereo:', len(stereo))
for f in stereo:
    x, sr = sf.read(os.path.join(AUDIO_DIR, f))
    kiri, kanan = x[:, 0], x[:, 1]
    beda = np.abs(kiri - kanan).max()
    puncak = np.abs(x).max()
    korelasi = np.corrcoef(kiri, kanan)[0, 1]
    print(f'{f} | selisih maks: {beda:.5f} | puncak sinyal: {puncak:.5f} | korelasi kiri-kanan: {korelasi:.4f}')

Mounted at /content/drive
Jumlah file stereo: 5
Actor_01/03-01-02-01-01-02-01.wav | selisih maks: 0.00000 | puncak sinyal: 0.03235 | korelasi kiri-kanan: 1.0000
Actor_01/03-01-08-01-02-02-01.wav | selisih maks: 0.00000 | puncak sinyal: 0.06747 | korelasi kiri-kanan: 1.0000
Actor_05/03-01-02-01-02-02-05.wav | selisih maks: 0.00000 | puncak sinyal: 0.02942 | korelasi kiri-kanan: 1.0000
Actor_20/03-01-03-01-02-01-20.wav | selisih maks: 0.00000 | puncak sinyal: 0.08972 | korelasi kiri-kanan: 1.0000
Actor_20/03-01-06-01-01-02-20.wav | selisih maks: 0.00000 | puncak sinyal: 0.03131 | korelasi kiri-kanan: 1.0000
